In [ ]:
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from matplotlib import pyplot as plt
from scipy import stats

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dust_jwst_highz.model.dust import (
    grain_size_dist,
    kappa_lambda,
    mass_absorption_coefficient,
    small_carbonaceous_grain_dist,
)

col_f = plt.get_cmap("gray")

DATA_DIR = Path("../data")

MICRON = 1e-4  # micron to cm
ANGSTROM = 1e-8  # angstrom to cm

In [ ]:
##############################################
##########    INPUT ARRAYS         ###########
##############################################


dust_pah = pd.read_csv(DATA_DIR / "dust_PAH.csv", comment="#")
dust_graphite = pd.read_csv(DATA_DIR / "dust_graphite.csv", comment="#")
dust_silicate = pd.read_csv(DATA_DIR / "dust_silicate.csv", comment="#")

with open(DATA_DIR / "wd01_mw_gsd_params.yaml") as f:
    WD01_MW_PARAMS = yaml.safe_load(f)

# -- NOW: show large grains are less UV absorbing than small grains
# -- (depending on grain size vs wavelength, specifically if a<<lambda or not)

# -----------------------------
# 2. Choose two grain sizes
# -----------------------------

radius_small = 0.001  # 0.001 micron, smallest in Draine table
radius_big = 0.1585  # 0.3 micron, where Hirashita young dust centred

# Densities
rho_carb = 2.24  # graphite
rho_sil = 3.5  # silicate

In [ ]:
# -----------------------------
# 3. Compute kappa_abs
# -----------------------------

# Carbonaceous
kappa_c_small = mass_absorption_coefficient(
    radius_small * MICRON,
    dust_graphite.loc[np.isclose(dust_graphite["radius"], radius_small), "Q_abs"],
    rho_carb,
)
kappa_c_big = mass_absorption_coefficient(
    radius_big * MICRON,
    dust_graphite.loc[np.isclose(dust_graphite["radius"], radius_big), "Q_abs"],
    rho_carb,
)

# Silicate
kappa_s_small = mass_absorption_coefficient(
    radius_small * MICRON,
    dust_silicate.loc[np.isclose(dust_silicate["radius"], radius_small), "Q_abs"],
    rho_sil,
)
kappa_s_big = mass_absorption_coefficient(
    radius_big * MICRON,
    dust_silicate.loc[np.isclose(dust_silicate["radius"], radius_big), "Q_abs"],
    rho_sil,
)

wavelength_c = np.sort(dust_graphite["wavelength"].unique())[::-1]
wavelength_s = np.sort(dust_silicate["wavelength"].unique())[::-1]

In [ ]:
# -----------------------------
# 4. Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 6))

# Carbonaceous: same color, different linestyle
ax.loglog(wavelength_c, kappa_c_small, color="indigo", ls="-", label=rf"Carbonaceous, $a={radius_small:.2e}\,\mu$m")
ax.loglog(wavelength_c, kappa_c_big, color="indigo", ls="--", label=rf"Carbonaceous, $a={radius_big:.2e}\,\mu$m")
# Silicate: same color, different linestyle
ax.loglog(wavelength_s, kappa_s_small, color="darkorange", ls="-", label=rf"Silicate, $a={radius_small:.2e}\,\mu$m")
ax.loglog(wavelength_s, kappa_s_big, color="darkorange", ls="--", label=rf"Silicate, $a={radius_big:.2e}\,\mu$m")

# Mark transition from Rayleigh to geometric (a ~ lambda), a smalla nd big are the same for silicate and carbon
# plt.axvline(a_small_c, color="grey",  alpha=0.5,lw=3, ls='-')
# plt.text(a_small_c*1.1, 10, r'$\lambda=a_{\rm small}$', rotation=90, color='grey', alpha=0.7, fontsize=12)
ax.axvline(radius_big, color="grey", alpha=0.6, lw=1.5, ls="--")
ax.text(radius_big * 1.1, 200, r"$\lambda=a_{\rm big}$", rotation=90, color="grey", alpha=0.7, fontsize=12)

# Asymptotic scalings
# geometric-limit κ = 3/(4 ρ a)   (convert a[µm]→cm: a*1e-4)
k_geo_s = 3 / (4 * rho_carb * (radius_small * 1e-4))
k_geo_b = 3 / (4 * rho_carb * (radius_big * 1e-4))

# Rayleigh scaling, normalized to meet at λ = a_s
k_ray = k_geo_s * (radius_small / wavelength_c)
ax.loglog(wavelength_c, k_ray, lw=3, label=r"Rayleigh $\propto \lambda^{-1}$", color="dimgrey", alpha=0.3)
# plt.hlines(k_geo_s, 1e-4, 1e3, color='grey', lw=2, label=r'geom (small)')
ax.hlines(k_geo_b, 1e-4, 1e3, lw=3, label=r"geom (big)", color="dimgrey", ls="--", alpha=0.3)
ax.axvline(0.1500, lw=6, color="indigo", alpha=0.1)


# Check power law behabiour in the FIR
lam_fir = np.linspace(20, 1000, 100)
ax.loglog(lam_fir, 12.85127581 * (lam_fir / 158.0) ** (-2), color="r", ls=":", lw=2, label=r"$\lambda^{-2}$ in FIR")


# -----------------------------
# 5. Formatting
# -----------------------------
ax.set_xlabel(r"$\lambda$ [µm]")
ax.set_ylabel(r"$\kappa_{\rm abs}\,$ [cm$^2$ g$^{-1}$]")
ax.legend(frameon=False, fontsize=13, loc="upper right")
# plt.grid(alpha=0.25)
ax.set_xlim(0.050, 30)
ax.set_ylim(1e2, 0.6e7)

In [ ]:
# choose MW model: R_V = 3.1 (normal), case A, b_C = 2.0 entry
rv = "3.1"
dcase = "A"
bc_idx = 2
bc = bc_idx * 1.0e-5
p = WD01_MW_PARAMS[rv][dcase][bc_idx]

# grain-size grid in cm
a_grid = np.logspace(np.log10(3.5e-8), np.log10(1e-4), 500)  # 3.5 Å – 1 µm

# your very–small–grain D(a) (per H); must accept a in cm
# e.g. D_of_a_MW(a_grid)


# Create graphite parameters with converted units (micron → cm)
graphite_params = p["graphite"].copy()
graphite_params["at"] *= MICRON
graphite_params["ac"] *= MICRON

graphite_dist = grain_size_dist(
    a_grid,
    **graphite_params,
    d_func=partial(small_carbonaceous_grain_dist, bc=bc),
)

# Create silicate parameters with converted units (micron → cm)
silicate_params = p["silicate"].copy()
silicate_params["at"] *= MICRON
silicate_params["ac"] = 0.1 * MICRON  # WD01 adopt a_c,s = 0.1 µm for MW

silicate_dist = grain_size_dist(
    a_grid,
    **silicate_params,
)

Yc = 1e29 * (a_grid**4) * graphite_dist  # for carbonaceous
Ys = 1e29 * (a_grid**4) * silicate_dist  # for silicates

plt.figure(figsize=(7, 5))
plt.loglog(a_grid * 1e4, Yc, color="indigo", label="Carbonaceous")
plt.loglog(a_grid * 1e4, Ys, color="darkorange", label="Silicate")

plt.xlabel(r"Grain radius $a\,[\mu{\rm m}]$")
plt.ylabel(r"$10^{29}\, a^{4} (n_{\rm H}^{-1} dn/da)$ [cm$^{3}$]")
plt.legend(frameon=False)
plt.grid(alpha=0.3)
plt.ylim(1e-2, 1e2)
plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------------------
# --- ok let's now compute opacity per unit dust mass
# -------------------------------------------------------------------

# ---- Carbon ----
radius_graphite = dust_graphite["radius"].unique() * MICRON

graphite_dist = grain_size_dist(
    radius_graphite,
    **graphite_params,
    d_func=partial(small_carbonaceous_grain_dist, bc=bc),
)
kappa_c_abs = kappa_lambda(
    radius_graphite,
    dust_graphite.pivot(index="radius", columns="wavelength", values="Q_abs").values,
    graphite_dist,
)
kappa_c_sca = kappa_lambda(
    radius_graphite,
    dust_graphite.pivot(index="radius", columns="wavelength", values="Q_sca").values,
    graphite_dist,
)


# ---- Silicate ----
radius_silicate = dust_silicate["radius"].unique() * MICRON

silicate_dist = grain_size_dist(
    radius_silicate,
    **silicate_params,
)
kappa_s_abs = kappa_lambda(
    radius_silicate,
    dust_silicate.pivot(index="radius", columns="wavelength", values="Q_abs").values,
    silicate_dist,
)
kappa_s_sca = kappa_lambda(
    radius_silicate,
    dust_silicate.pivot(index="radius", columns="wavelength", values="Q_sca").values,
    silicate_dist,
)

# ---- Total MW mixture (C+Si) ----
kappa_abs_tot = (1.0 / 11.0) * kappa_c_abs + (10.0 / 11.0) * kappa_s_abs
kappa_sca_tot = (1.0 / 11.0) * kappa_c_sca + (10.0 / 11.0) * kappa_s_sca

kappa_ext_tot = kappa_abs_tot + kappa_sca_tot  # true extinction
omega_tot = kappa_sca_tot / kappa_ext_tot  # albedo ω(λ)


wavelength = dust_graphite["wavelength"].unique()[::-1] * 1e4  # µm → Å


plt.figure(figsize=(7, 5))

plt.loglog(wavelength, kappa_c_abs, color="indigo", alpha=0.6, label="Carbonaceous")
plt.loglog(wavelength, kappa_s_abs, color="darkorange", alpha=0.6, label="Silicate")
plt.loglog(wavelength, kappa_abs_tot, color="black", lw=2, label="Total MW")
plt.plot(wavelength, kappa_ext_tot, color="black", lw=2, ls="--", label="Total MW (with scattering)")


print("kext_1500 [cm^2/g] -->", kappa_ext_tot[wavelength == 1585])  # tweak as needed
kuv_drn = kappa_ext_tot[wavelength == 1585]

plt.xlabel(r"Wavelength $\lambda$ [$\AA$]", fontsize=14)
plt.ylabel(r"$\kappa_{\rm abs}(\lambda)\ [{\rm cm}^2\,{\rm g}^{-1}]$", fontsize=14)
plt.legend(frameon=False, fontsize=12)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ------ Ok now we add the case of the readily produced from stars dust (Hirashita=19) GSD -----#
# --- lognormal GSD centred at 0.3 micron with sigma=0.1

PROTON_MASS = 1.67262192e-24  # Proton mass [g]
MEAN_MOL_WEIGHT = 1.22  # Mean molecular weight (typical ISM)
DR_MW = 1.0 / 162.0  # Milky Way dust-to-gas ratio


# def C_phi_lognormal(a0_cm=1e-5, sigma=0.47):
#     """Normalization constant C_phi for the lognormal grain size
#     distribution of stellar dust (eq. 5 in your screenshot).

#     a0_cm : float
#         Central grain radius a0 in cm (0.1 micron → 1e-5 cm).
#     sigma : float
#         Lognormal width (dimensionless).
#     """
#     pref = (4.0 / 3.0) * np.pi * a0_cm**3
#     return 1.0 / (pref * np.sqrt(2.0 * np.pi) * sigma * np.exp(0.5 * 9.0 * sigma**2))


# def stellar_grain_size_dist(a_cm, a0_cm=1e-5, sigma=0.47):
#     """Lognormal stellar grain size distribution φ(a) such that

#         ∫ (4π/3) a^3 φ(a) da = 1  (mass-normalized).

#     Parameters
#     ----------
#     a_cm : array_like
#         Grain radius in cm.
#     a0_cm : float
#         Central grain radius in cm (default 0.1 μm = 1e-5 cm).
#     sigma : float
#         Lognormal width.

#     Returns
#     -------
#     phi : ndarray
#         φ(a) with the same shape as a_cm.
#         Units: [1 / (cm^4)] so that (4π/3) ∫ a^3 φ(a) da is dimensionless.

#     """
#     a_cm = np.asarray(a_cm, dtype=float)
#     Cphi = C_phi_lognormal(a0_cm=a0_cm, sigma=sigma)

#     x = np.log(a_cm / a0_cm)
#     return Cphi / a_cm * np.exp(-(x**2) / (2.0 * sigma**2))


def stellar_grain_size_dist(a_cm, a0_cm=1e-5, sigma=0.47):
    """Lognormal stellar grain size distribution.

    φ(a) such that

        ∫ (4π/3) a^3 φ(a) da = 1  (mass-normalized).

    Uses scipy.stats.lognorm for the underlying distribution.

    Parameters
    ----------
    a_cm : array_like
        Grain radius in cm.
    a0_cm : float
        Central grain radius in cm (default 0.1 μm = 1e-5 cm).
    sigma : float
        Lognormal width (standard deviation of log(a)).

    Returns
    -------
    phi : ndarray
        φ(a) with the same shape as a_cm.
        Units: [1 / (cm^4)] so that (4π/3) ∫ a^3 φ(a) da is dimensionless.

    """
    a_cm = np.asarray(a_cm, dtype=float)

    # scipy lognorm: PDF(x) = 1/(x*sigma*sqrt(2*pi)) * exp(-(ln(x) - ln(scale))^2 / (2*sigma^2))
    # We set scale=a0_cm so the distribution is centered at a0_cm
    lognorm_pdf = stats.lognorm.pdf(a_cm, s=sigma, scale=a0_cm)

    # Normalize so that ∫ (4π/3) a^3 φ(a) da = 1
    # The normalization constant accounts for the mass weighting
    volume_factor = (4.0 / 3.0) * np.pi * a0_cm**3
    correction_factor = np.exp(4.5 * sigma**2)  # from ∫ a^3 lognorm(a) da

    return lognorm_pdf / (volume_factor * correction_factor)


a_grid_cm = np.logspace(-7, -4, 500)  # 0.001–1 μm
phi = stellar_grain_size_dist(a_grid_cm, a0_cm=1e-5, sigma=0.47)

# Check normalization numerically:
mass_int = (4 * np.pi / 3) * np.trapezoid(a_grid_cm**3 * phi, a_grid_cm)
# print("should be 1 (cause norm) -->", mass_int)


# --- Now compare WD01 GSD with stellar lognormal GSD ---#
# stellar lognormal φ(a) on those grids (mass-normalized)
phi_graphite_star = stellar_grain_size_dist(radius_graphite, a0_cm=1e-5, sigma=0.47)  # a0 = 0.1 μm
phi_silicate_star = stellar_grain_size_dist(radius_silicate, a0_cm=1e-5, sigma=0.47)

# convert φ(a) → (1/n_H) dn/da with the same D as MW
factor_grapite = MEAN_MOL_WEIGHT * PROTON_MASS * DR_MW / ((4.0 * np.pi / 3.0) * rho_carb)
factor_silicate = MEAN_MOL_WEIGHT * PROTON_MASS * DR_MW / ((4.0 * np.pi / 3.0) * rho_sil)

graphite_dist_star = factor_grapite * phi_graphite_star  # same units as dn_da_C
silicate_dist_star = factor_silicate * phi_silicate_star  # same units as dn_da_Si

# -------- Draine-style comparison plot --------
plt.figure(figsize=(7, 5))

# WD01 carbonaceous
plt.loglog(
    radius_graphite * 1e4, 1e29 * radius_graphite**4 * graphite_dist, color="indigo", lw=2, label="WD01 - Carbonaceous"
)
# WD01 silicate
plt.loglog(
    radius_silicate * 1e4, 1e29 * radius_silicate**4 * silicate_dist, color="darkorange", lw=2, label="WD01 - Silicate"
)

# Stellar lognormal carbonaceous with same D (this is independent from species)
plt.loglog(
    radius_graphite * 1e4,
    1e29 * radius_graphite**4 * graphite_dist_star,
    color="indigo",
    ls="--",
    lw=2,
    label="Stellar ",
)


plt.xlabel(r"$a\ [\mu{\rm m}]$", fontsize=14)
plt.ylabel(r"$10^{29}\,a^4\,{\rm d}n/{\rm d}a\ [{\rm cm}^3]$", fontsize=14)
plt.legend(frameon=False)
plt.grid(alpha=0.3)
plt.ylim(1e-2, 1e2)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# 2. κ_abs and κ_sca for stellar dust
# ------------------------------------------------------------------
# --- Carbonaceous ---
kappa_c_abs_star = kappa_lambda(
    radius_graphite,
    dust_graphite.pivot(index="radius", columns="wavelength", values="Q_abs").values,
    graphite_dist_star,
)
kappa_c_sca_star = kappa_lambda(
    radius_graphite,
    dust_graphite.pivot(index="radius", columns="wavelength", values="Q_sca").values,
    graphite_dist_star,
)

# --- Silicate ---
kappa_s_abs_star = kappa_lambda(
    radius_silicate,
    dust_silicate.pivot(index="radius", columns="wavelength", values="Q_abs").values,
    silicate_dist,
)
kappa_s_sca_star = kappa_lambda(
    radius_silicate,
    dust_silicate.pivot(index="radius", columns="wavelength", values="Q_sca").values,
    silicate_dist,
)

# put both components on the same λ grid (they *should* match; if not, interp)
lam_um = wavelength_c  # = wavelength_s
# Total stellar mixture
kappa_abs_star_tot = kappa_c_abs_star + kappa_s_abs_star
kappa_sca_star_tot = kappa_c_sca_star + kappa_s_sca_star

kappa_ext_star_tot = kappa_abs_star_tot + kappa_sca_star_tot
omega_star_tot = kappa_sca_star_tot / kappa_ext_star_tot  # albedo


print("kext_1500_stellar [cm^2/g] -->", kappa_ext_star_tot[wavelength == 1585])
kuv_hir = kappa_ext_star_tot[wavelength == 1585]

# ------------------------------------------------------------------
# 3. (Optional) compare stellar vs WD01 MW
# ------------------------------------------------------------------


plt.figure(figsize=(7, 5))
plt.loglog(wavelength, kappa_abs_tot, "k--", lw=1, label="MW WD01 (abs)")
plt.loglog(wavelength, kappa_ext_tot, "k", lw=1.5, label="MW WD01 (tot)")
plt.loglog(wavelength, kappa_abs_star_tot, "r--", lw=1, label="Stellar (abs)")
plt.loglog(wavelength, kappa_ext_star_tot, "r", lw=1.5, label="Stellar (tot)")
plt.xlim(923, 2e4)
plt.ylim(1e2, 1e6)
plt.xlabel(r"$\lambda\ [\AA]$", fontsize=14)
plt.ylabel(r"$\kappa_{\lambda}\ [{\rm cm}^2\,{\rm g}^{-1}]$", fontsize=14)
plt.legend(frameon=False, fontsize=14)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# -- Ok now that I ma not retarded anymore I can redo Fig. 5 in the paper ----#

# ==========================================================
# Compare MW WD01 vs stellar dust: κ_λ + a^4 dn/da inset
# ==========================================================

fig, ax = plt.subplots(figsize=(10, 10))

# --- MAIN PANEL: κ_ext(λ) ---
ax.plot(wavelength, kappa_ext_tot, lw=2.5, alpha=0.5, color="crimson", label="MW dust (WD01)")
ax.plot(wavelength, kappa_ext_star_tot, lw=2.0, alpha=0.8, color="teal", label="Stellar dust (H19)")
# -- abs
ax.plot(wavelength, kappa_abs_tot, lw=2.5, alpha=0.5, color="crimson", ls=":")
ax.plot(wavelength, kappa_abs_star_tot, lw=2.0, alpha=0.8, color="teal", ls=":")


print("\n==== κ_abs(1500 Å) results ====")
print(f"MW WD01 dust (ext)     : {kuv_drn} cm^2 g^-1")
print(f"Stellar dust model (ext): {kuv_hir} cm^2 g^-1")
print(f"WD01 dust (abs)     : {kappa_abs_tot[wavelength == 1.585e3]} cm^2 g^-1")
print(f"Stellar dust model (abs): {kappa_abs_star_tot[wavelength == 1.585e3]} cm^2 g^-1")
print("================================\n")

# --- Plot Points at 1500 Å ---
ax.scatter(
    1500,
    kuv_drn,
    color="crimson",
    edgecolor="black",
    s=70,
    marker="D",
    alpha=0.4,
    label=rf"$\kappa_{{1500}}={int(kuv_drn[0])}$ $cm^2\,g^{{-1}}$ (MW WD01)",
)

ax.scatter(
    1500,
    kuv_hir,
    color="teal",
    edgecolor="black",
    s=70,
    marker="D",
    alpha=0.4,
    label=rf"$\kappa_{{1500}}={int(kuv_hir[0])}$ $cm^2\,g^{{-1}}$ (Stellar)",
)

# vertical line at 1500 Å
ax.axvline(1500.0, color="gray", linestyle="--", lw=1)


ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(25, 2e5)
ax.set_xlim(1e3, 2e6)  # 1000 Å – 200 µm

ax.text(1550.0, ax.get_ylim()[0] * 1.2, r"$\lambda=1500\,$Å", fontsize=12, color="gray")


ax.set_xlabel(r"$\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\kappa_{\lambda}\ [{\rm cm}^2\,{\rm g}^{-1}]$")
ax.grid(True, alpha=0.2, lw=0.5)
ax.legend(frameon=False, fontsize=16, loc="lower right")

# ==========================================================
# INSET: 1e29 * a^4 dn/da for WD01 vs stellar dust
# ==========================================================
inset_ax = ax.inset_axes([0.57, 0.57, 0.4, 0.4])

# build a common size grid (cm) for total (C+Si)

a_all = np.sort(np.unique(np.concatenate([radius_graphite, radius_silicate])))

# interpolate WD01 dn/da (per H) onto common grid
dn_da_wd = np.interp(a_all, radius_graphite, graphite_dist, left=0.0, right=0.0)
dn_da_wd += np.interp(a_all, radius_silicate, silicate_dist, left=0.0, right=0.0)
# interpolate stellar dn/da onto same grid
dn_da_star = np.interp(a_all, radius_graphite, graphite_dist_star, left=0.0, right=0.0)
dn_da_star += np.interp(a_all, radius_silicate, silicate_dist_star, left=0.0, right=0.0)

# quantity to plot: 1e29 * a^4 dn/da   [cm^3]
y_wd = 1e29 * a_all**4 * dn_da_wd
y_star = 1e29 * a_all**4 * dn_da_star

# convert a to µm for x–axis
a_um_all = a_all * 1e4

inset_ax.loglog(a_um_all, y_wd, color="crimson", alpha=0.7, lw=2, label="MW dust")
inset_ax.loglog(a_um_all, y_star, color="teal", alpha=0.8, lw=2, label="Stellar dust (H19)")

inset_ax.set_xscale("log")
inset_ax.set_yscale("log")
inset_ax.set_xlim(1e-3, 1.0)
# pick a sensible y–range; can tweak after first run
inset_ax.set_ylim(1e-3, 1e3)

inset_ax.set_xlabel(r"$a\ [\mu{\rm m}]$", fontsize=14)
inset_ax.set_ylabel(r"$10^{29}\,a^4\,{\rm d}n/{\rm d}a\ [{\rm cm}^3]$", fontsize=14)
inset_ax.grid(True, which="both", ls=":", alpha=0.2)


kir_drn = kappa_abs_tot[wavelength == 1.585e6]
kir_hir = kappa_abs_star_tot[wavelength == 1.585e6]


print("\n==== κ_abs(158 micron) results (for dust emission) ====")
print(f"MW WD01 dust (abs)     : {kir_drn} cm^2 g^-1")
print(f"Stellar dust model (abs): {kir_hir} cm^2 g^-1")
print("=========================================================\n")

kv_drn = kappa_ext_tot[wavelength == 5.623e3]
kv_hir = kappa_ext_star_tot[wavelength == 5.623e3]

print("\n==== κ_tot(V band) results ====")
print(f"MW WD01 dust (ext)     : {kv_drn} cm^2 g^-1")
print(f"Stellar dust model (ext): {kv_hir} cm^2 g^-1")
print("=========================================================\n")

In [ ]:
radius_graphite, radius_silicate

In [ ]:
a_all

In [ ]:
import astropy.units as apu

apu.angstrom.find_equivalent_units()